# PBT #1 checkpoint: evaluation and inference video

Loads **only the rank-01** checkpoint under `best_pbt_checkpoints_top5/` (best eval_score in the manifest), runs a **deterministic** evaluation aligned with training (`evaluate_current_model` in `ray_pbt_train.py`), then shows one **inference rollout** as an inline animation (same rendering pattern as `VideoProgressCallback._record_and_show` in `unit1 - Lunar Lander agent_v3.ipynb`).

**Eval:** For **each** value in **`SEEDS`** (10 seeds by default), a **separate** eval vector env (`make_eval_vec_env_synced`): same `VecNormalize` hyperparameters and synced observation stats as the loaded training env (`training=False`, `norm_reward=False` — no reward scaling). The underlying env is `make_lunar_env` (**`Monitor`** + native `LunarLander-v3` rewards and episode limits — no extra reward or horizon wrappers). **`evaluate_policy`** with **`N_EVAL_EPISODES`**, **`deterministic=True`**. Prints **`mean_reward +/- std_reward`** and leaderboard-style **mean − std** per seed. The eval env is closed in `evaluate_current_model` (`try` / `finally`). `best_eval_score_so_far` in `trainer_state.json` refers to PBT-period eval and may differ.

**Video:** optional inline animation — the rollout + `display_rollout_video` block at the end of the eval cell is **commented out** by default; uncomment to render.

**Hub (optional):** run `notebook_login()` once, then the `package_to_hub` cell — it loads the **same rank-01** checkpoint, builds `eval_env` with `make_eval_vec_env_synced` (same as training), and pushes like Unit 1. Edit `repo_id` / `model_name` in that cell.

**Setup:** open this notebook from the repo root (or set `REPO_ROOT` in the next cell). **`SEEDS`** in the imports cell lists **10** eval env seeds (default `35..44`); the eval cell loops over them and prints metrics each time. **`SEED = SEEDS[0]`** is kept for the Hub / optional video. Use the project `.venv` like the Unit 1 notebook. For headless servers, install `xvfb` and use `pyvirtualdisplay` as in Unit 1 so `rgb_array` rendering works.

In [51]:
import os
import warnings

# Silence pygame → setuptools pkg_resources deprecation (main process + SubprocVecEnv workers).
os.environ.setdefault("PYTHONWARNINGS", "ignore::UserWarning:pygame.pkgdata")
warnings.filterwarnings(
    "ignore",
    message=".*pkg_resources is deprecated.*",
    category=UserWarning,
    module="pygame.pkgdata",
)

%matplotlib inline

import json
import sys
from pathlib import Path

import matplotlib.animation as animation
import matplotlib.pyplot as plt
from IPython.display import HTML, display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "ray_pbt_train.py").is_file():
    raise RuntimeError(
        "Set the notebook working directory to the RL-LunarLander repo root "
        "(File → Open Folder), or set REPO_ROOT manually."
    )
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from lunar_rl_common import DEFAULT_ENV_ID, make_eval_vec_env_synced
from ray_pbt_train import evaluate_current_model, load_trial_checkpoint

CHECKPOINT_ROOT = REPO_ROOT / "best_pbt_checkpoints_top5"

# Eval env seeds (loop in eval cell); overrides trainer_state.json seed per run.
SEEDS = list(range(35, 45))  # 10 seeds: 35..44 — edit as needed
SEED = SEEDS[0]  # Hub cell / optional video

In [52]:
def sorted_checkpoint_dirs(root: Path) -> list[Path]:
    dirs = [p for p in root.iterdir() if p.is_dir() and (p / "model.zip").is_file()]
    return sorted(dirs, key=lambda p: p.name)


def collect_rollout_frames(model, train_venv, seed: int, env_id: str | None):
    """One deterministic episode; RGB frames via env.render (matches VideoProgressCallback)."""
    eval_venv = make_eval_vec_env_synced(train_venv, seed, env_id)
    try:
        eval_venv.seed(seed)
        obs = eval_venv.reset()
        frames = []
        fr = eval_venv.env_method("render")[0]
        if fr is not None:
            frames.append(fr)
        total_reward = 0.0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, info = eval_venv.step(action)
            total_reward += float(reward[0])
            fr = eval_venv.env_method("render")[0]
            if fr is not None:
                frames.append(fr)
            if done[0]:
                break
        return frames, total_reward
    finally:
        eval_venv.close()


def display_rollout_video(frames, title: str, interval_ms: int = 33) -> None:
    if not frames:
        print(f"{title}: no frames (check display / rgb_array rendering).")
        return
    fig, ax = plt.subplots(figsize=(6, 4), dpi=72)
    ax.axis("off")
    fig.subplots_adjust(left=0, right=1, top=0.9, bottom=0.05)
    fig.suptitle(title, fontsize=11)
    im = ax.imshow(frames[0])

    def update(i):
        im.set_array(frames[i])
        return (im,)

    anim = animation.FuncAnimation(
        fig, update, frames=len(frames), interval=interval_ms, blit=False
    )
    html = anim.to_jshtml()
    plt.close(fig)
    display(HTML(html))


_all = sorted_checkpoint_dirs(CHECKPOINT_ROOT)
checkpoint_dirs = _all[:1]
if not checkpoint_dirs:
    raise RuntimeError(f"No checkpoint with model.zip under {CHECKPOINT_ROOT}")
print(
    f"Using rank #1 only ({checkpoint_dirs[0].name}); skipped {len(_all) - 1} other(s)."
)

# Pooled eval: same protocol as PBT (eval_seeds × periodic_eval_episodes from merged["base"]).
for ckpt_dir in checkpoint_dirs:
    print("\n" + "=" * 72)
    print(ckpt_dir.name)
    state_path = ckpt_dir / "trainer_state.json"
    with open(state_path, encoding="utf-8") as f:
        tr_state = json.load(f)
    merged = tr_state["current_config"]
    stored_best = tr_state.get("best_eval_score_so_far")

    model, train_env, _loaded = load_trial_checkpoint(
        str(ckpt_dir), merged, env_id=DEFAULT_ENV_ID
    )
    try:
        if stored_best is not None:
            print(
                f"  trainer_state.json best_eval_score_so_far (during PBT): "
                f"{float(stored_best):.2f}"
            )
        print(
            "  (VecNormalize synced obs; norm_reward=False; training=False.)"
        )
        base = merged["base"]
        es = base.get("eval_seeds") or list(DEFAULT_EVAL_SEEDS)
        n_per = int(base.get("periodic_eval_episodes", 10))
        total_eps = len(es) * n_per
        mean_r, std_r, score = evaluate_current_model(
            model, train_env, merged["base"], env_id=DEFAULT_ENV_ID
        )
        print(
            f"\n  Pooled eval ({len(es)} seeds × {n_per} eps = {total_eps} total, deterministic):"
        )
        print(f"    mean_reward={mean_r:.2f} +/- {std_r:.2f}")
        print(f"    eval_score (mean - std, leaderboard)={score:.2f}")
        # Video replay — uncomment the block below to run rollout + inline animation.
        # frames, ep_ret = collect_rollout_frames(
        #     model, train_env, seed, DEFAULT_ENV_ID
        # )
        # print(
        #     f"video replay (1 episode only — not the eval score above): return={ep_ret:.2f}"
        # )
        # display_rollout_video(frames, title=ckpt_dir.name)
    finally:
        train_env.close()

Using rank #1 only (rank01_ts5963776_mean301.37_eval293.77_trial1e06f_00012_srccheckpoint_000009); skipped 4 other(s).

rank01_ts5963776_mean301.37_eval293.77_trial1e06f_00012_srccheckpoint_000009
  trainer_state.json best_eval_score_so_far (during PBT): 293.77
  (VecNormalize synced obs; norm_reward=False; training=False.)

  --- SEED 35 ---
  evaluate_policy (n_eval_episodes=10, deterministic=True):
    mean_reward=286.99 +/- 14.56
    eval_score (mean - std, leaderboard)=272.43

  --- SEED 36 ---
  evaluate_policy (n_eval_episodes=10, deterministic=True):
    mean_reward=301.12 +/- 16.60
    eval_score (mean - std, leaderboard)=284.52

  --- SEED 37 ---
  evaluate_policy (n_eval_episodes=10, deterministic=True):
    mean_reward=293.14 +/- 19.01
    eval_score (mean - std, leaderboard)=274.13

  --- SEED 38 ---
  evaluate_policy (n_eval_episodes=10, deterministic=True):
    mean_reward=288.60 +/- 15.45
    eval_score (mean - std, leaderboard)=273.15

  --- SEED 39 ---
  evaluate_poli

In [53]:
import subprocess

from huggingface_hub import notebook_login

notebook_login()
subprocess.run(["git", "config", "--global", "credential.helper", "store"], check=False)

CompletedProcess(args=['git', 'config', '--global', 'credential.helper', 'store'], returncode=0)

In [54]:
import inspect
import json
from pathlib import Path

from huggingface_hub import ModelCard
from huggingface_sb3 import package_to_hub

from lunar_rl_common import DEFAULT_ENV_ID, make_eval_vec_env_synced
from ray_pbt_train import load_trial_checkpoint

import stable_baselines3.common.vec_env.vec_normalize as _vn

def _safe_vecnormalize_getstate(self):
    state = self.__dict__.copy()
    state.pop("venv", None)
    state.pop("class_attributes", None)
    state.pop("returns", None)
    return state

_vn.VecNormalize.__getstate__ = _safe_vecnormalize_getstate

# Rank-01 checkpoint: first dir under CHECKPOINT_ROOT with model.zip
def _checkpoint_dirs(root: Path):
    dirs = [p for p in root.iterdir() if p.is_dir() and (p / "model.zip").is_file()]
    return sorted(dirs, key=lambda p: p.name)

# --- Hub settings (edit repo_id / names as needed) ---
repo_id = "ntitz19/ppo-LunarLander-v3"
model_name = "ppo-LunarLander-v3"
model_architecture = "PPO"
commit_message = "Upload PPO LunarLander-v3 MlpPolicy vector-obs agent (PBT checkpoint)"
env_id = DEFAULT_ENV_ID

ckpt_list = _checkpoint_dirs(CHECKPOINT_ROOT)[:1]
if not ckpt_list:
    raise RuntimeError(f"No checkpoint with model.zip under {CHECKPOINT_ROOT}")
ckpt_dir = ckpt_list[0]

with open(ckpt_dir / "trainer_state.json", encoding="utf-8") as f:
    tr_state = json.load(f)
merged = tr_state["current_config"]
model, train_env, _loaded = load_trial_checkpoint(
    str(ckpt_dir), merged, env_id=env_id
)
eval_env = None
try:
    eval_env = make_eval_vec_env_synced(train_env, SEED, env_id=env_id)
    _sig = inspect.signature(package_to_hub)
    _kwargs = dict(
        model=model,
        model_name=model_name,
        model_architecture=model_architecture,
        env_id=env_id,
        eval_env=eval_env,
        repo_id=repo_id,
        commit_message=commit_message,
    )
    if "n_eval_episodes" in _sig.parameters:
        _kwargs["n_eval_episodes"] = 200

    package_to_hub(**_kwargs)
finally:
    if eval_env is not None:
        eval_env.close()
    train_env.close()

card = ModelCard.load(repo_id)
if "LunarLander-v2" not in card.data.tags:
    card.data.tags.append("LunarLander-v2")
    card.push_to_hub(
        repo_id, commit_message="Add LunarLander-v2 tag for course certification compatibility"
    )


ℹ This function will save, evaluate, generate a video of your agent,
create a model card and push everything to the hub. It might take up to 1min.
This is a work in progress: if you encounter a bug, please open an issue.
Saving video to /tmp/tmpir4gwhww/-step-0-to-step-1000.mp4
MoviePy - Building video /tmp/tmpir4gwhww/-step-0-to-step-1000.mp4.
MoviePy - Writing video /tmp/tmpir4gwhww/-step-0-to-step-1000.mp4



MoviePy - Done !
MoviePy - video ready /tmp/tmpir4gwhww/-step-0-to-step-1000.mp4
✘ 'DummyVecEnv' object has no attribute 'video_recorder'
✘ We are unable to generate a replay of your agent, the package_to_hub
process continues
✘ Please open an issue at
https://github.com/huggingface/huggingface_sb3/issues
ℹ Pushing repo ntitz19/ppo-LunarLander-v3 to the Hugging Face Hub


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

ℹ Your model is pushed to the Hub. You can view your model here:
https://huggingface.co/ntitz19/ppo-LunarLander-v3/tree/main/
